# Fast Particle Simulation Analysis

This notebook analyzes Monte Carlo particle simulation data using ROOT framework. It focuses on visualizing and characterizing particle trajectories and properties for different particle types (muons, pions, and protons).

## Main objectives:

1. **Spatial visualization**: Generate plane views (XY and ZY projections) of particle trajectories in the detector geometry
2. **Particle properties**: Analyze and compare initial conditions and characteristics across different particle species:
    - Transverse momentum (pT)
    - Track length (L)
    - Lever Arm length (LArm) 
    - Number of measurement points (N)

The analysis processes data from fast particle simulations, creating publication-quality plots saved in EPS and PNG formats in the `Properties/` directory.

### Loading packages and setting up the environment

```python

In [ ]:
import os
import ROOT
from Plot_func import SetHisto
from Plot_func import SetCanvas
from Plot_func import SetLegend
from Plot_func import SetGlobalStyle
from Plot_func import SetColor
from ROOT import  gROOT, gStyle, gSystem
gSystem.Load("../aliKalman/AliExternalTrackParam.so")
gROOT.LoadMacro("../MC/fastSimulation.cxx+")
gROOT.LoadMacro("../MC/fastSimulationTest.C")

In [ ]:
folder = "../data/"
foldercheck="Properties/"

os.makedirs(foldercheck, exist_ok=True)
inputData = folder+"fastParticle.list"
tree  = ROOT.AliXRDPROOFtoolkit.MakeChainRandom(inputData,"fastPart",chr(0),10000)

# Plane Views

In [ ]:
tree.SetAlias("gxMC","cos(part.fParamMC[0].fAlpha)*part.fParamMC[0].fX")
tree.SetAlias("gyMC","sin(part.fParamMC[0].fAlpha)*part.fParamMC[0].fX")
tree.SetAlias("gzMC","part.fParamMC[0].fP[1]")


SetColor()
SetGlobalStyle()

gStyle.SetOptStat(0)
hq = ROOT.TCanvas("hq","hq",600,600)
SetCanvas(hq)
hq.SetLeftMargin(0.2)
hq.SetRightMargin(0.2)
hq.SetTopMargin(0.2)
hq.SetBottomMargin(0.2)
tree.Draw("gyMC:gxMC>>hxy(100,-320,320,100,-320,320)","part.fParamMC[0].fP[4]!=0","colz")
h=ROOT.gDirectory.Get("hxy")
SetHisto(h,";#it{x} (cm);#it{y} (cm)",ROOT.kBlack,20,[-320,320],False)
h.GetZaxis().SetTitleOffset(1.6)
h.GetZaxis().SetTitle("Entries (a.u.)")
el1 = ROOT.TEllipse(0,0,250,250)
el1.SetFillStyle(0)
el1.Draw("same")
el1.SetLineWidth(3)
hq.Draw()
hq.Print(foldercheck+"XY_view.eps")
hq.Print(foldercheck+"XY_view.png")


hq2 = ROOT.TCanvas("hq2","hq2",600,600)
SetCanvas(hq2)
hq2.SetLeftMargin(0.2)
hq2.SetRightMargin(0.2)
hq2.SetTopMargin(0.2)
hq2.SetBottomMargin(0.2)
tree.Draw("gyMC:gzMC>>hzy(100,-320,320,100,-320,320)","part.fParamMC[0].fP[4]!=0","colz")
hzy=ROOT.gDirectory.Get("hzy")
hzy.GetZaxis().SetTitle("Entries (a.u.)")
hzy.GetZaxis().SetTitleOffset(1.6)
SetHisto(hzy,";#it{z} (cm);#it{y} (cm)",ROOT.kBlack,20,[-320,320],False)
box1 = ROOT.TBox(-250,-250,250,250)
box1.SetFillStyle(0)
box1.SetLineWidth(3)
box1.Draw("same")
hq2.Draw()
hq2.Print(foldercheck+"ZY_view.eps")
hq2.Print(foldercheck+"ZY_view.png")

# Particle initial conditions

## LArm, NPoints, p, l

In [ ]:
def PropPlot(t1,var,varname,histprop,histheight,cond,canvas,legendchi2,letter="(a)"):
    SetColor()
    SetGlobalStyle()

   
    SetCanvas(canvas)
    canvas.SetLeftMargin(0.19)
    canvas.SetBottomMargin(0.16)
    t1.Draw(var+">>hpT("+histprop+")",cond[0])
    hpT=ROOT.gDirectory.Get("hpT")
    SetHisto(hpT,";"+varname+";Entries (a.u.)",ROOT.kBlue,20,[0,histheight])
    hpT.SetLineColor(ROOT.kBlue)
    hpT.GetXaxis().SetTitleSize(0.07)
    hpT.GetXaxis().SetTitleOffset(1.1)
    hpT.GetYaxis().SetTitleSize(0.07)
    hpT.GetYaxis().SetTitleOffset(1.4)
    hpT.GetXaxis().SetLabelSize(0.07)
    hpT.GetYaxis().SetLabelSize(0.07)


    t1.Draw(var+">>hpT211("+histprop+")",cond[1])
    hpT211=ROOT.gDirectory.Get("hpT211")
    SetHisto(hpT211,";"+varname+";Entries (a.u.)",ROOT.kGreen,20,[0,histheight])
    hpT211.SetLineColor(ROOT.kGreen) 
    t1.Draw(var+">>hpT2212("+histprop+")",cond[2])
    hpT2212=ROOT.gDirectory.Get("hpT2212")
    SetHisto(hpT2212,";"+varname+";Entries (a.u.)",ROOT.kRed,20,[0,histheight])
    hpT2212.SetLineColor(ROOT.kRed)


    hpT.Draw("E HIST")
    hpT211.Draw("E HIST SAME")
    hpT2212.Draw("E HIST SAME")
    #legendchi2 = ROOT.TLegend(0.44,0.7,0.92,0.88)
    SetLegend(legendchi2)
    legendchi2.SetTextSize(0.05*0.68*2)
    legendchi2.SetHeader(letter)
    legendchi2.AddEntry(hpT,"Muons")
    legendchi2.AddEntry(hpT211,"Pions")
    legendchi2.AddEntry(hpT2212,"Protons")



In [ ]:
hq1 = ROOT.TCanvas("hq1","hq1",900,1200)
legendchi2 = ROOT.TLegend(0.44,0.7,0.92,0.88)
varname= "#it{p}_{T} (GeV/#it{c})"
histox = "40,0,5"
histoy = 80
cond = ["part.fParamMC[0].fP[4]!=0 && abs(pdgCode)==13",
        "part.fParamMC[0].fP[4]!=0 && abs(pdgCode)==211",
        "part.fParamMC[0].fP[4]!=0 && abs(pdgCode)==2212"]

PropPlot(tree,"abs(1/part.fParamMC[0].fP[4])",varname,histox,histoy,cond,hq1,legendchi2,"(a)")
legendchi2.Draw()
hq1.Draw()
hq1.Print(foldercheck+"pTAllTall.eps")
hq1.Print(foldercheck+"pTAllTall.png")

In [ ]:
hq1 = ROOT.TCanvas("hq1","hq1",900,1200)
legendchi2 = ROOT.TLegend(0.44,0.7,0.92,0.88)
varname = "#it{L} (cm)"
histox = "40,0,600"
histoy = 60

PropPlot(tree,"Length",varname,histox,histoy,cond,hq1,legendchi2,"(a)")
legendchi2.Draw()
hq1.Draw()
hq1.Print(foldercheck+"LAllTall.eps")
hq1.Print(foldercheck+"LAllTall.png")

In [ ]:
hq1 = ROOT.TCanvas("hq1","hq1",900,1200)
legendchi2 = ROOT.TLegend(0.44,0.7,0.92,0.88)
varname = "#it{N}"
histox = "40,0,600"
histoy = 60
PropPlot(tree,"part.fParamMC@.size()",varname,histox,histoy,cond,hq1,legendchi2,"(c)")
legendchi2.Draw()
hq1.Draw()
hq1.Print(foldercheck+"NTall.eps")
hq1.Print(foldercheck+"NTall.png")

In [ ]:
tree.SetAlias("gxMCSt","(part.fParamMC[0].fX*cos(part.fParamMC[0].fAlpha)-part.fParamMC[0].fP[0]*sin(part.fParamMC[0].fAlpha))")
tree.SetAlias("gyMCSt","(part.fParamMC[0].fX*sin(part.fParamMC[0].fAlpha)+part.fParamMC[0].fP[0]*cos(part.fParamMC[0].fAlpha))")
tree.SetAlias("gxMCEnd","(part.fParamMC[part.fParamMC@.size()-1].fX*cos(part.fParamMC[part.fParamMC@.size()-1].fAlpha)-part.fParamMC[part.fParamMC@.size()-1].fP[0]*sin(part.fParamMC[part.fParamMC@.size()-1].fAlpha))")
tree.SetAlias("gyMCEnd","(part.fParamMC[part.fParamMC@.size()-1].fX*sin(part.fParamMC[part.fParamMC@.size()-1].fAlpha)+part.fParamMC[part.fParamMC@.size()-1].fP[0]*cos(part.fParamMC[part.fParamMC@.size()-1].fAlpha))")
tree.SetAlias("LArmMC","sqrt((gxMCSt-gxMCEnd)*(gxMCSt-gxMCEnd)+(gyMCSt-gyMCEnd)*(gyMCSt-gyMCEnd))")

hq1 = ROOT.TCanvas("hq1","hq1",900,1200)
legendchi2 = ROOT.TLegend(0.44,0.7,0.92,0.88)
varname = "#it{L}_{Arm} (cm)"
histox = "40,0,600"
histoy = 60
PropPlot(tree,"LArmMC",varname,histox,histoy,cond,hq1,legendchi2,"(b)")
legendchi2.Draw()
hq1.Draw()
hq1.Print(foldercheck+"LArmTall.eps")
hq1.Print(foldercheck+"LArmTall.png")